# 07_deploy — register the champion and batch-score active projects

1. **Pick** the champion feature set by **validation** AUC (never by test AUC — choosing on test would make the test score optimistic).
2. **Register** that model in the Unity Catalog model registry as `dbw_wb_outcome.ml.project_outcome_xgb` and point the alias **`champion`** at the new version.
3. **Batch-score** currently active projects (approved FY2019+, not yet rated) into `gold.active_project_scores`.

**Why batch, not a serving endpoint:** projects are approved a few times a week and nobody needs a prediction in milliseconds; a scheduled batch job costs minutes of cluster time, while an always-on endpoint bills around the clock.

**Attach to:** `wb-ml-single`.

## Settings

**What:** registry location (`databricks-uc`), model name, experiment path.

**Why:** setting the registry URI to Unity Catalog makes the model a governed UC object — versioned, permissioned, and lineage-linked to the tables it was trained on.

In [1]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F

CATALOG = "dbw_wb_outcome"
MODEL_NAME = f"{CATALOG}.ml.project_outcome_xgb"
ME = spark.sql("select current_user()").first()[0]
EXPERIMENT = f"/Users/{ME}/wb-outcome/wb_outcome_experiment"
from databricks.sdk import WorkspaceClient
WorkspaceClient().workspace.mkdirs(f"/Users/{ME}/wb-outcome")   # the experiment's parent folder must exist
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

## 1. Choose the champion by validation AUC

**What:** reads the latest parent run of each feature set and picks the highest `val_auc`.

**Why:** model selection is part of training, so it must use validation data. The test score then stays an honest estimate of how the chosen model will do on future projects.

In [2]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="tags.stage = 'ablation'",
                          order_by=["start_time DESC"])
runs = runs[runs["tags.mlflow.parentRunId"].isna()] if "tags.mlflow.parentRunId" in runs else runs
latest = runs.drop_duplicates("tags.feature_set")
best = latest.sort_values("metrics.val_auc", ascending=False).iloc[0]
print(latest[["tags.feature_set", "metrics.val_auc", "metrics.test_auc"]].to_string(index=False))
print("champion:", best["tags.feature_set"], "| run", best["run_id"])

tags.feature_set  metrics.val_auc  metrics.test_auc
        combined         0.606588          0.588785
            text         0.578700          0.575844
      structured         0.613124          0.601671
champion: structured | run 6c1b2852aedf44f0bca65f1730e26b82


## 2. Register in Unity Catalog and set the `champion` alias

**What:** registers the run's logged model as a new version of `ml.project_outcome_xgb`, adds a description and tags, and moves the `champion` alias to it.

**Why:** consumers load `models:/…@champion`, never a hard-coded version number. Promoting a new model = moving the alias; rolling back = moving it back. No consumer code changes.

In [3]:
mv = mlflow.register_model(best["tags.model_uri"], MODEL_NAME)
client.update_registered_model(MODEL_NAME, description=(
    "Average of 3 independently tuned XGBoost classifiers (MLflow pyfunc): "
    "P(World Bank project rated Moderately Satisfactory or better by IEG). "
    "Features known at Board approval only: project attributes, country indicators lagged 1 year, "
    "and pre-approval document text. Trained FY1995-2011, tested FY2012+."))
client.update_model_version(MODEL_NAME, mv.version, description=(
    f"feature_set={best['tags.feature_set']}; val_auc={best['metrics.val_auc']:.3f}; "
    f"test_auc={best['metrics.test_auc']:.3f}; source run {best['run_id']}"))
for k, v in {"feature_set": best["tags.feature_set"], "val_auc": f"{best['metrics.val_auc']:.4f}",
             "test_auc": f"{best['metrics.test_auc']:.4f}"}.items():
    client.set_model_version_tag(MODEL_NAME, mv.version, k, v)
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"registered {MODEL_NAME} v{mv.version} -> @champion")

Registered model 'dbw_wb_outcome.ml.project_outcome_xgb' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '7' of model 'dbw_wb_outcome.ml.project_outcome_xgb': https://<workspace-host>/explore/data/models/dbw_wb_outcome/ml/project_outcome_xgb/version/7?o=<workspace-id>


registered dbw_wb_outcome.ml.project_outcome_xgb v7 -> @champion


## 3. Batch-score active projects with `@champion`

**What:** loads the model by alias (a pyfunc that averages 3 tuned XGBoost models), selects exactly the feature columns it stores, predicts P(success) for every `split = 'score'` project, and ranks them by risk of failure. The top 20% are flagged `High` risk.

**Why:** reading the column list from the model itself guarantees training/scoring consistency. A ranked list with a risk band is what a portfolio manager would actually act on — "review these 20% first".

In [4]:
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
feature_cols = model.unwrap_python_model().features

score = spark.table(f"{CATALOG}.gold.project_features").filter("split = 'score'").toPandas()
score["prob_success"] = np.asarray(model.predict(score[feature_cols].astype("float64")))
score["risk_of_failure"] = 1 - score["prob_success"]
score["risk_rank"] = score["risk_of_failure"].rank(ascending=False, method="first").astype(int)
cut = score["risk_of_failure"].quantile(0.8)
score["risk_band"] = (score["risk_of_failure"] >= cut).map({True: "High", False: "Normal"})
score["model_version"] = int(mv.version)
score["scored_at"] = datetime.now(timezone.utc)

out = score[["project_id", "project_name", "country_name", "approval_date", "approval_fy",
             "prob_success", "risk_of_failure", "risk_rank", "risk_band", "model_version", "scored_at"]]
spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.active_project_scores")

display(spark.table(f"{CATALOG}.gold.active_project_scores").orderBy("risk_rank").limit(15))

project_id,project_name,country_name,approval_date,approval_fy,prob_success,risk_of_failure,risk_rank,risk_band,model_version,scored_at
P177372,Gabon Urban Development Project,Gabon,2025-06-11,2025,0.5809,0.4191,1,High,7,2026-10-16T16:47:42.557361Z
P505271,Republic of Congo - Poultry and Aquaculture Development Project,"Congo, Republic of",2026-08-07,2027,0.5978,0.4022,2,High,7,2026-10-16T16:47:42.557361Z
P179190,Republic of Congo Strengthening Urban Resilience Project,"Congo, Republic of",2025-10-30,2026,0.6015,0.3985,3,High,7,2026-10-16T16:47:42.557361Z
P161877,DRC Transport and Connectivity Support Project,"Congo, Democratic Republic of",2022-06-23,2022,0.6062,0.3938,4,High,7,2026-10-16T16:47:42.557361Z
P168772,Valorization of Investments in the Valley of the Logone,Cameroon,2021-11-30,2022,0.6094,0.3906,5,High,7,2026-10-16T16:47:42.557361Z
P171658,Bamako Urban Resilience project,Mali,2022-11-30,2023,0.6108,0.3892,6,High,7,2026-10-16T16:47:42.557361Z
P172422,Senegal Cadastre and Land Tenure Improvement Project,Senegal,2021-06-15,2021,0.6168,0.3832,7,High,7,2026-10-16T16:47:42.557361Z
P171933,Ghana Landscape Restoration and Small-Scale Mining Project,Ghana,2021-08-30,2022,0.618,0.382,8,High,7,2026-10-16T16:47:42.557361Z
P164330,Greater Accra Resilient and Integrated Development Project,Ghana,2019-05-29,2019,0.6192,0.3808,9,High,7,2026-10-16T16:47:42.557361Z
P180017,"Cameroon Second Fiscal Sustainability, Inclusive and Sustainable Growth Development Policy Financing",Cameroon,2025-03-14,2025,0.6194,0.3806,10,High,7,2026-10-16T16:47:42.557361Z


## 4. Summary

**What:** the registered version and scoring counts as the notebook exit value.

**Why:** lets the scheduled workflow report what was deployed without opening the notebook.

In [5]:
summary = {"model": MODEL_NAME, "version": int(mv.version), "feature_set": best["tags.feature_set"],
           "val_auc": round(float(best["metrics.val_auc"]), 4), "test_auc": round(float(best["metrics.test_auc"]), 4),
           "scored_projects": len(out), "high_risk": int((out.risk_band == "High").sum())}
print(summary)
dbutils.notebook.exit(json.dumps(summary))

{"model": "dbw_wb_outcome.ml.project_outcome_xgb", "version": 7, "feature_set": "structured", "val_auc": 0.6131, "test_auc": 0.6017, "scored_projects": 2505, "high_risk": 501}